#### Preprocessing the datasets!
*author*: Gabriela Klima

This notebook loads in the three different datasets, checks if any cleaning is necessary, and merges the 3 datasets together so that it is ready for visualisations in the streamlit app.

In [136]:
# Imports and directories
import pandas as pd
import numpy as np
from pathlib import Path

raw_dir = Path('datasets') # where raw datasets are

# dataset files ~
pollution_file = raw_dir / 'air_pollutant_emissions.csv'
expenditure_file = raw_dir / 'health_expenditure.csv'
deaths_file = raw_dir / 'premature_deaths_fine_particles.csv'

Inspecting the Air Pollutant Emissions file,

tab-separated, UTF-16 encoding, numerical values use commas for format, one column per year!

In [137]:
air_raw = pd.read_csv(pollution_file, sep='\t', encoding='utf-16')

print(f'Shape: {air_raw.shape}')
print(f'Columns: {list(air_raw.columns)[:8]}')

air_raw.head()

Shape: (204, 39)
Columns: ['Country Name', 'Pollutant', 'Order EEA Sector', 'EEA Sector', 'Unit', '1990', '1991', '1992']


,Country Name,Pollutant,Order EEA Sector,EEA Sector,Unit,1990,1991,1992,1993,1994,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,EU27,NH3,8,National total for the entire territory (based...,kt,"5,285","4,999","4,732","4,579","4,487",...,"3,825","3,854","3,866","3,865","3,799","3,702","3,648","3,572","3,372","3,381"
1,EU27,NMVOC,8,National total for the entire territory (based...,kt,"16,433","15,735","15,079","14,540","13,510",...,"7,166","7,158","6,981","7,001","6,869","6,748","6,666","6,614","6,380","6,153"
2,EU27,NOx,8,National total for the entire territory (based...,kt,"15,088","14,846","14,416","13,849","13,316",...,"7,337","7,276","6,978","6,898","6,597","6,277","5,542","5,660","5,335","5,052"
3,EU27,PM2.5,8,National total for the entire territory (based...,kt,"2,716","2,699","2,560","2,596","2,426",...,"1,534","1,549","1,508","1,486","1,517","1,418","1,334","1,371","1,249","1,170"
4,EU27,PM10,8,National total for the entire territory (based...,kt,"4,041","4,014","3,713","3,707","3,497",...,"2,268","2,270","2,210","2,188","2,245","2,118","2,010","2,061","1,915","1,825"


In [138]:
# Checking values for countries, pollutants as well as sector and units

print(f'Countries: {sorted(air_raw['Country Name'].unique())}')
print(f'Pollutants: {sorted(air_raw['Pollutant'].unique())}')
print(f'Units {air_raw["Unit"].unique()}')
print(f'Sectors: {air_raw['EEA Sector'].unique()}')

Countries: ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'EU27', 'EU32', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Latvia', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Türkiye']
Pollutants: ['NH3', 'NMVOC', 'NOx', 'PM10', 'PM2.5', 'SOx']
Units <StringArray>
['kt']
Length: 1, dtype: str
Sectors: <StringArray>
['National total for the entire territory (based on fuel sold)']
Length: 1, dtype: str


In [139]:
# All units and Sectors are the same so we can drop them
air_df = air_raw.drop(columns=['Order EEA Sector', 'EEA Sector', 'Unit']).copy()

# Dropping EU aggregates so that the df is per country only. (Keeping all the rows where country name is NOT EU27/EU32)
air_df = air_df[~air_df['Country Name'].isin(['EU27', 'EU32'])]

# Converting the year columns from string to numeric (no commas!!)
year_cols = [c for c in air_df.columns if c.isdigit()]
for c in year_cols:
    air_df[c] = pd.to_numeric(air_df[c].astype(str).str.replace(',', ''), errors='coerce')

# Checking missing values (summing up missing values in each column, then summing up those numbers)
missing_total = air_df[year_cols].isna().sum().sum()
print(f'Missing values across all years: {missing_total}')
print(f'Shape after cleaning: {air_df.shape}')

Missing values across all years: 0
Shape after cleaning: (192, 36)


In [140]:
air_pivot = air_df.melt(
    id_vars=['Country Name','Pollutant'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Emissions_kt'
)

air_pivot['Year'] = air_pivot['Year'].astype(int)
air_pivot = air_pivot.rename(columns={'Country Name': 'Country'})

print(f'New shape {air_pivot.shape}')
air_pivot.head()

New shape (6528, 4)


,Country,Pollutant,Year,Emissions_kt
0,Austria,NH3,1990,90
1,Austria,NMVOC,1990,338
2,Austria,NOx,1990,216
3,Austria,PM2.5,1990,27
4,Austria,PM10,1990,43


Made it too long, need to fix table so that country, year is in a single row... (Extracting each pollutant out of 'Pollutant')

In [141]:
# Taking the long dataset and creating one row for every country/year combination WITH a new column for each pollutant (values from Emissions_kt)
air_clean = air_pivot.pivot_table(
    index=['Country', 'Year'],
    columns='Pollutant',
    values='Emissions_kt'
).reset_index()
air_clean.columns.name = None # fixing pandas pivot table column axis name

# Dictionary; looping through each pollutant type, replacing any '.' to '_' and adding '_kt' at the end. e.g. 'NH3': 'NH3_kt'
rename = {p: f'{p.replace(".", "_")}_kt' for p in ['NH3', 'NMVOC', 'NOx', 'PM10', 'PM2.5', 'SOx']}

# Using the dictionary to rename the columns
air_clean = air_clean.rename(columns=rename)
print(f'Final shape: {air_clean.shape}')
air_clean.head()

Final shape: (1088, 8)


,Country,Year,NH3_kt,NMVOC_kt,NOx_kt,PM10_kt,PM2_5_kt,SOx_kt
0,Austria,1990,90.0,338.0,216.0,43.0,27.0,74.0
1,Austria,1991,90.0,334.0,226.0,43.0,27.0,71.0
2,Austria,1992,87.0,310.0,214.0,43.0,27.0,54.0
3,Austria,1993,87.0,291.0,208.0,43.0,27.0,53.0
4,Austria,1994,86.0,268.0,200.0,42.0,26.0,47.0


Exploring Premature Deaths from fine particles csv.

Already in geo_level~country/year/metric format - Just have to stick to country level!

In [142]:
deaths_raw = pd.read_csv(deaths_file)

print(f'Shape: {deaths_raw.shape}')
print(f'Columns: {list(deaths_raw.columns)}')
deaths_raw.head()

Shape: (57096, 10)
Columns: ['code', 'dimension', 'dimension_label', 'unit', 'unit_label', 'geo', 'geo_label', 'time', 'obs_value', 'obs_status']


,code,dimension,dimension_label,unit,unit_label,geo,geo_label,time,obs_value,obs_status
0,11_52,PMD,Premature deaths - Premature deaths,NR,Number,AL0,SHQIPËRIA,2023,3551,NaN
1,11_52,YLL,Premature deaths - Years of life lost,NR,Number,AL0,SHQIPËRIA,2015,49506,NaN
2,11_52,YLL,Premature deaths - Years of life lost,NR,Number,AL0,SHQIPËRIA,2021,40017,NaN
3,11_52,YLL,Premature deaths - Years of life lost,NR,Number,AL0,SHQIPËRIA,2012,49597,NaN
4,11_52,PMD,Premature deaths - Premature deaths,NR,Number,AL0,SHQIPËRIA,2020,3792,NaN


In [143]:
# Keeping only 3-char geo-codes (NUTS-1 region / country totals)
deaths_nuts1 = deaths_raw[deaths_raw['geo'].str.len() == 3].copy()
deaths_nuts1['country_iso2'] = deaths_nuts1['geo'].str[:2]

# aggregating to national totals
deaths_country = (
    # min_count=1 keeps Nan values instead of returning 0
    deaths_nuts1.groupby(['country_iso2', 'dimension', 'time'])['obs_value'].sum(min_count=1).reset_index()
)

# pivoting PMD/YLL to columns
deaths_clean = deaths_country.pivot_table(
    index=['country_iso2', 'time'],
    columns='dimension',
    values='obs_value'
).reset_index()
deaths_clean.columns.name = None

# Renaming columns
deaths_clean = deaths_clean.rename(columns={
    'time': 'Year',
    'PMD': 'Premature_deaths',
    'YLL': 'Years_of_life_lost'
})

print(f'Country Count: {deaths_clean['country_iso2'].nunique()}')
print(f'List of country codes {deaths_clean['country_iso2'].unique()}')
print(f'Cleaned shape: {deaths_clean.shape}')
deaths_clean.head()


Country Count: 35
List of country codes <StringArray>
['AL', 'AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI',
 'FR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LI', 'LT', 'LU', 'LV', 'ME', 'MK', 'MT',
 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK']
Length: 35, dtype: str
Cleaned shape: (630, 4)


,country_iso2,Year,Premature_deaths,Years_of_life_lost
0,AL,2005,6269.0,65343.0
1,AL,2007,4844.0,52251.0
2,AL,2008,4280.0,46337.0
3,AL,2009,5099.0,54474.0
4,AL,2010,6031.0,64986.0


In [144]:
# Cleaning Country Column
deaths_clean = deaths_clean.rename(columns={
    'country_iso2': 'Country'
})

rename_countries = {
    'AL': 'Albania', 'AT': 'Austria',
    'BE': 'Belgium', 'BG': 'Bulgaria',
    'CH': 'Switzerland', 'CY': 'Cyprus', 'CZ': 'Czechia',
    'DE': 'Germany', 'DK': 'Denmark',
    'EE': 'Estonia', 'EL': 'Greece', 'ES': 'Spain',
    'FI': 'Finland', 'FR': 'France',
    'HR': 'Croatia', 'HU': 'Hungary',
    'IE': 'Ireland', 'IS': 'Iceland', 'IT': 'Italy',
    'LI': 'Liechtenstein', 'LT': 'Lithuania', 'LU': 'Luxembourg', 'LV': 'Latvia',
    'ME': 'Montenegro', 'MK': 'North Macedonia', 'MT': 'Malta',
    'NL': 'Netherlands', 'NO': 'Norway',
    'PL': 'Poland', 'PT': 'Portugal',
    'RO': 'Romania', 'RS': 'Serbia',
    'SE': 'Sweden', 'SI': 'Slovenia', 'SK': 'Slovakia'
}

deaths_clean['Country'] = deaths_clean['Country'].replace(rename_countries)

print(f'list of countries now: {list(deaths_clean['Country'].unique())}')

list of countries now: ['Albania', 'Austria', 'Belgium', 'Bulgaria', 'Switzerland', 'Cyprus', 'Czechia', 'Germany', 'Denmark', 'Estonia', 'Greece', 'Spain', 'Finland', 'France', 'Croatia', 'Hungary', 'Ireland', 'Iceland', 'Italy', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Latvia', 'Montenegro', 'North Macedonia', 'Malta', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'Romania', 'Serbia', 'Sweden', 'Slovenia', 'Slovakia']


In [145]:
# Checking for missing values
deaths_clean.isna().sum()

Country               0
Year                  0
Premature_deaths      0
Years_of_life_lost    0
dtype: int64

Exploring Health Expenditure csv~

Has 4 metadata header rows before actual table, wide format, includes regions and non-european countries.

In [146]:
health_raw = pd.read_csv(expenditure_file, skiprows=4)

print(f'list of columns: {list(health_raw.columns)}')
print(f'Original shape: {health_raw.shape}')
health_raw.head()

list of columns: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70']
Original shape: (266, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,89.867260,89.969369,87.039877,81.446140,92.350333,92.035288,88.485585,NaN,NaN,NaN
2,Afghanistan,AFG,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,66.823883,71.225090,74.064239,80.089233,81.521126,80.651604,59.067707,NaN,NaN,NaN
3,Africa Western and Central,AFW,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,58.509817,56.286041,59.267134,64.624190,72.632850,74.441253,65.886854,NaN,NaN,NaN
4,Angola,AGO,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,112.753647,83.875557,64.446609,56.016418,66.944870,103.945061,76.071182,NaN,NaN,NaN


In [147]:
health_raw = health_raw.drop(columns=[c for c in health_raw.columns if c.startswith('Unnamed')])
print(f'shape: {health_raw.shape}')
print(f'Indicators: {health_raw['Indicator Name'].unique()}')

shape: (266, 70)
Indicators: <StringArray>
['Current health expenditure per capita (current US$)']
Length: 1, dtype: str


In [148]:
# Renaming Countries as before, removing unnecessary countries

ios3_rename = {
    'ALB': 'Albania', 'AUT': 'Austria', 'BEL': 'Belgium', 'BGR': 'Bulgaria',
    'CHE': 'Switzerland', 'CYP': 'Cyprus', 'CZE': 'Czechia', 'DEU': 'Germany',
    'DNK': 'Denmark', 'EST': 'Estonia', 'GRC': 'Greece', 'ESP': 'Spain',
    'FIN': 'Finland', 'FRA': 'France', 'HRV': 'Croatia', 'HUN': 'Hungary',
    'IRL': 'Ireland', 'ISL': 'Iceland', 'ITA': 'Italy', 'LIE': 'Liechtenstein',
    'LTU': 'Lithuania', 'LUX': 'Luxembourg', 'LVA': 'Latvia', 'MNE': 'Montenegro',
    'MKD': 'North Macedonia', 'MLT': 'Malta', 'NLD': 'Netherlands', 'NOR': 'Norway',
    'POL': 'Poland', 'PRT': 'Portugal', 'ROU': 'Romania', 'SRB': 'Serbia',
    'SWE': 'Sweden', 'SVN': 'Slovenia', 'SVK': 'Slovakia', 'TUR': 'Türkiye'
}

health_eu = health_raw[health_raw['Country Code'].isin(ios3_rename.keys())].copy()
health_eu['Country'] = health_eu['Country Code'].map(ios3_rename)

print(f'list of country names now: {list(health_eu['Country'].unique())}')
health_eu.head()

list of country names now: ['Albania', 'Austria', 'Belgium', 'Bulgaria', 'Switzerland', 'Cyprus', 'Czechia', 'Germany', 'Denmark', 'Spain', 'Estonia', 'Finland', 'France', 'Greece', 'Croatia', 'Hungary', 'Ireland', 'Iceland', 'Italy', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Latvia', 'North Macedonia', 'Malta', 'Montenegro', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'Romania', 'Serbia', 'Slovakia', 'Slovenia', 'Sweden', 'Türkiye']


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Country
5,Albania,ALB,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,295.523468,349.211273,366.340149,396.174774,465.570435,506.869202,590.627991,NaN,NaN,Albania
14,Austria,AUT,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,4999.869629,5394.559082,5313.192871,5569.841309,6554.142578,5897.976074,6267.931641,6739.309082,NaN,Austria
17,Belgium,BEL,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,4827.593262,5233.682617,5087.456543,5269.406250,5847.173340,5487.592773,5931.014648,NaN,NaN,Belgium
21,Bulgaria,BGR,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,647.564026,719.780334,733.068787,906.548340,1109.102905,1066.691895,1257.875977,NaN,NaN,Bulgaria
37,Switzerland,CHE,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,9354.220703,9541.074219,9609.802734,10294.578125,11197.764648,10930.139648,11783.669922,NaN,NaN,Switzerland


In [149]:
health_year_cols = [c for c in health_eu.columns if c.isdigit()]
health_long = health_eu.melt(
    id_vars=['Country'],
    value_vars=health_year_cols,
    var_name='Year',
    value_name='Health_expenditure_per_capita_USD'
)
health_long['Year'] = health_long['Year'].astype(int)

health_clean = health_long.dropna(subset=['Health_expenditure_per_capita_USD']).reset_index(drop=True)

print(f'Cleaned shape: {health_clean.shape}')
health_clean.head()

Cleaned shape: (856, 3)


,Country,Year,Health_expenditure_per_capita_USD
0,Albania,2000,65.476746
1,Austria,2000,2301.892090
2,Belgium,2000,1849.916016
3,Bulgaria,2000,94.519760
4,Switzerland,2000,3550.125977


Merging into a single dataset

outer join all three on (Country, Year)

In [150]:
merged = (
    air_clean
    .merge(deaths_clean, on=['Country', 'Year'], how='outer')
    .merge(health_clean, on=['Country', 'Year'], how='outer')
)

merged = merged.sort_values(['Country', 'Year']).reset_index(drop=True)
print(f'Merged shape: {merged.shape}')
print(f'Countries: {merged['Country'].nunique()}')
merged.head(10)

Merged shape: (1194, 11)
Countries: 36


,Country,Year,NH3_kt,NMVOC_kt,NOx_kt,PM10_kt,PM2_5_kt,SOx_kt,Premature_deaths,Years_of_life_lost,Health_expenditure_per_capita_USD
0,Albania,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.476746
1,Albania,2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.921715
2,Albania,2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78.729095
3,Albania,2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,105.436676
4,Albania,2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136.507248
5,Albania,2005,NaN,NaN,NaN,NaN,NaN,NaN,6269.0,65343.0,149.975159
6,Albania,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,164.220901
7,Albania,2007,NaN,NaN,NaN,NaN,NaN,NaN,4844.0,52251.0,209.319107
8,Albania,2008,NaN,NaN,NaN,NaN,NaN,NaN,4280.0,46337.0,237.103409
9,Albania,2009,NaN,NaN,NaN,NaN,NaN,NaN,5099.0,54474.0,236.942444


The common window for all 3 datasets is between 2005-2023. Will drop rows where the year is less than 2005 or bigger than 2023.

In [151]:
merged = merged[merged['Year'].between(2005, 2023)]

merged.head(3)

,Country,Year,NH3_kt,NMVOC_kt,NOx_kt,PM10_kt,PM2_5_kt,SOx_kt,Premature_deaths,Years_of_life_lost,Health_expenditure_per_capita_USD
5,Albania,2005,NaN,NaN,NaN,NaN,NaN,NaN,6269.0,65343.0,149.975159
6,Albania,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,164.220901
7,Albania,2007,NaN,NaN,NaN,NaN,NaN,NaN,4844.0,52251.0,209.319107


In [152]:
# Checking missingness per column
missing_by_country = merged.isna().groupby(merged['Country']).sum()
missing_by_country

,Country,Year,NH3_kt,NMVOC_kt,NOx_kt,PM10_kt,PM2_5_kt,SOx_kt,Premature_deaths,Years_of_life_lost,Health_expenditure_per_capita_USD
Country,,,,,,,,,,,
Albania,0,0,19,19,19,19,19,19,1,1,0
Austria,0,0,0,0,0,0,0,0,1,1,0
Belgium,0,0,0,0,0,0,0,0,1,1,0
Bulgaria,0,0,0,0,0,0,0,0,1,1,0
Croatia,0,0,0,0,0,0,0,0,1,1,0
Cyprus,0,0,0,0,0,0,0,0,1,1,0
Czechia,0,0,0,0,0,0,0,0,1,1,0
Denmark,0,0,0,0,0,0,0,0,1,1,0
Estonia,0,0,0,0,0,0,0,0,1,1,0


Based on the missing values, Turkiye, Serbia, North Macedonia, Montenegro and Albania have 18+ years of data missing for some columns. I will drop these countries

In [153]:
drop_countries = ['Türkiye', 'Serbia', 'North Macedonia', 'Montenegro', 'Albania']

merged_clean = merged[~merged['Country'].isin(drop_countries)]

In [157]:
merged_clean.to_csv('datasets/processed/sustainability_merged.csv', index=False)